In [ ]:
# ========== 安装依赖：在 Colab / 环境里装好本周作业要用的库 ==========
# transformers：Hugging Face 模型与 pipeline（文本生成、分词等）
# diffusers：扩散模型（Stable Diffusion）图像生成
# accelerate：加速 / 设备映射辅助
# gradio：快速搭交互式 Web UI
# datasets：数据集工具（本作业主要间接依赖）
# --quiet：减少 pip 刷屏；逻辑未改，仅注释

!pip install transformers diffusers accelerate gradio datasets --quiet


In [54]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# torch：张量与 CUDA 设备检测（GPU / CPU）
import torch
# gradio：搭「合成数据集工作室」Web 界面
import gradio as gr
# random：与 torch 一起设随机种子，方便复现
import random
# json：解析模型输出的 JSON 文本
import json
# re：用正则从杂乱文本里抠出 {...} JSON 片段
import re
# pandas：把 JSON 列表转成 DataFrame / 导出 CSV
import pandas as pd
# pipeline：一行拉起文本生成；AutoTokenizer：按模型加载分词器（Tokenizer）
from transformers import pipeline, AutoTokenizer
# DiffusionPipeline：加载 Stable Diffusion 一类图像生成管线
from diffusers import DiffusionPipeline
# login / whoami：Hugging Face Hub 登录与身份（本格主要用 login）
from huggingface_hub import login, whoami
# userdata：从 Google Colab Secrets 读密钥，避免写死在笔记本里
from google.colab import userdata


In [ ]:
# ========== 可复现性（Reproducibility）：固定随机种子 ==========

# torch 侧随机种子：影响 CUDA / 张量随机采样
torch.manual_seed(42)
# Python random 侧种子：影响 random 模块的随机数
random.seed(42)


In [52]:
# ========== Hugging Face 登录：从 Colab Secrets 取 token ==========

try:
    # 从 Colab 密钥库读取 HF_API_KEY（需在 Secrets 里事先配置）
    hf_token = userdata.get("HF_API_KEY")
    if hf_token:
        # 用 token 登录 Hub，以便拉取 gated / 私有模型
        login(hf_token)
        # 给人看的状态信息：保留英文原样（不改运行时文案）
        print("Logged into Hugging Face")
    else:
        # 没找到密钥时的提示文案（依赖程序判断/原输出，不翻译）
        print("HF_TOKEN not found. Make sure to set it in Colab Secrets.")
except Exception as e:
    # 非 Colab 或登录失败时跳过，不中断后续单元格
    print("Login skipped or failed:", e)


Logged into Hugging Face


In [ ]:
# ========== 设备选择：有 GPU 用 GPU，否则 CPU ==========

# transformers pipeline 的 device：0 表示第一块 CUDA GPU，-1 表示 CPU
device = 0 if torch.cuda.is_available() else -1
# 打印当前走 GPU 还是 CPU（英文状态文案保持原样）
print("Using GPU" if device == 0 else "Using CPU")


Using CPU


In [ ]:
# ========== 模型注册表 + 懒加载：需要时再 pipeline ==========

# 展示名 → Hugging Face model id（这些字符串是可运行 model id，禁止改译）
models = {
    "FLAN-T5": "google/flan-t5-base",
    "DistilGPT2": "distilgpt2",
    "TinyLlama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
}

# 缓存已加载的 pipeline，避免重复下载/初始化
loaded_models = {}

def load_model(model_name):
    # 首次请求某个展示名时才真正加载
    if model_name not in loaded_models:
        # text-generation pipeline：统一接口做续写/生成；device 用上一格的 GPU/CPU 选择
        loaded_models[model_name] = pipeline(
            "text-generation",
            model=models[model_name],
            device=device
        )
    # 返回缓存中的 pipeline 实例
    return loaded_models[model_name]


In [ ]:
# ========== JSON 校验：把模型杂文尽量收成 dict ==========

def extract_json(text):
    try:
        # 理想情况：整段就是合法 JSON，直接 loads
        return json.loads(text)
    except:
        # 退路：用正则在文本里找第一段 {...}（DOTALL 让 . 能跨行）
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            try:
                # 对抠出的子串再试一次 JSON 解析
                return json.loads(match.group())
            except:
                # 找到了花括号但内容仍非法
                return {"error": "Invalid JSON format"}
        # 连花括号都没有
        return {"error": "No JSON detected"}


In [50]:
# ========== 合成文本数据：按任务类型循环生成并校验 JSON ==========

def generate_synthetic_data(task_type, model_name, num_samples):

    # 按所选展示名懒加载文本生成 pipeline
    pipe = load_model(model_name)

    # 任务类型 → 英文 prompt（发给模型的指令，禁止改译，改了会改变生成行为）
    prompts = {
        "Customer Support":
        """
Generate ONE realistic customer support interaction as VALID JSON only.

Return:
{
  "customer_message": "...",
  "agent_response": "...",
  "issue_category": "...",
  "resolution_status": "resolved or escalated",
  "customer_sentiment": "positive, neutral, or negative"
}
        """,

        "Product Reviews":
        """
Generate ONE realistic product review as VALID JSON only.

Return:
{
  "product_name": "...",
  "category": "...",
  "rating": 1-5,
  "review_title": "...",
  "review_text": "...",
  "verified_purchase": true or false,
  "sentiment": "positive, neutral, or negative"
}
        """,

        "Meeting Summary":
        """
Generate ONE structured meeting summary as VALID JSON only.

Return:
{
  "title": "...",
  "date": "YYYY-MM-DD",
  "participants": ["name1", "name2"],
  "key_points": ["point1", "point2"],
  "decisions_made": ["decision1"],
  "action_items": [
    {"task": "...", "owner": "...", "deadline": "YYYY-MM-DD"}
  ]
}
        """,

        "QA Dataset":
        """
Generate ONE QA training example as VALID JSON only.

Return:
{
  "domain": "...",
  "difficulty": "easy, medium, or hard",
  "context": "...",
  "question": "...",
  "answer": "...",
  "answer_type": "fact, explanation, or reasoning"
}
        """
    }

    # 收集本轮所有样本（校验后的 dict）
    results = []

    # num_samples 可能来自 Gradio Slider，先 int() 再循环
    for _ in range(int(num_samples)):
        # 调 pipeline：采样生成；取第一条结果的 generated_text
        output = pipe(
            prompts[task_type],
            max_new_tokens=180,
            do_sample=True,
            temperature=0.7
        )[0]["generated_text"]

        # 把模型原文尽量收成 JSON dict（失败则带 error 字段）
        validated = extract_json(output)
        # 追加到结果列表
        results.append(validated)

    # 返回样本列表，供 UI / CSV 导出使用
    return results


In [ ]:
# ========== JSON → DataFrame：方便表格查看与 to_csv ==========

def json_to_dataframe(json_data):
    try:
        # 列表[dict] → pandas.DataFrame（列对齐各 JSON 字段）
        return pd.DataFrame(json_data)
    except:
        # 转换失败时仍返回带 error 列的 DataFrame，避免 UI 崩溃
        return pd.DataFrame({"error": ["Conversion failed"]})


In [ ]:
# ========== 分词器检查：看一段文本被切成哪些 token ==========

def inspect_tokenizer(model_name, sample_text):
    # 按所选模型 id 加载对应 AutoTokenizer（与生成模型同一套词表）
    tokenizer = AutoTokenizer.from_pretrained(models[model_name])
    # tokenize：字符串 → token 字符串列表（如子词）
    tokens = tokenizer.tokenize(sample_text)
    # convert_tokens_to_ids：token 字符串 → 词表整数 id
    token_ids = tokenizer.convert_tokens_to_ids(tokens)

    # 打包给 Gradio JSON 组件展示
    return {
        "tokens": tokens,
        "token_ids": token_ids,
        "num_tokens": len(tokens)
    }


In [53]:
# ========== 加载扩散模型：有 GPU 才启用图像生成 ==========

if torch.cuda.is_available():
    # 从 Hub 拉 Stable Diffusion v1.5；float16 省显存
    image_pipe = DiffusionPipeline.from_pretrained(
        "stable-diffusion-v1-5/stable-diffusion-v1-5",
        torch_dtype=torch.float16
    ).to("cuda")

    # 注意力切片：用时间换显存，降低 OOM 风险
    image_pipe.enable_attention_slicing()
    # 部分权重可卸到 CPU，进一步省 GPU 显存
    image_pipe.enable_model_cpu_offload()

    print("Diffusion model loaded.")
else:
    # 无 GPU：禁用图像模式，后续 generate 会返回提示
    print("GPU not available. Image mode disabled.")
    image_pipe = None


GPU not available. Diffusion image mode disabled.


In [ ]:
# ========== 合成图像：用同一 prompt 连生成多张 ==========

def generate_synthetic_image_dataset(prompt, num_images):

    # 上一格若未加载成功，直接返回说明列表（文案保持原样）
    if image_pipe is None:
        return ["GPU not available — image generation disabled."]

    # 收集 PIL.Image 列表，交给 Gradio Gallery
    images = []
    for _ in range(int(num_images)):
        # 调用扩散管线；.images[0] 取第一张输出图
        image = image_pipe(prompt).images[0]
        images.append(image)

    return images


In [ ]:
# ========== Gradio UI：文本合成 + 分词检查 + 图像合成 ==========

def run_generator(task_type, model_name, num_samples):
    # 按 UI 选择生成并校验 JSON 样本
    data = generate_synthetic_data(task_type, model_name, num_samples)
    # 转成表格，便于导出
    df = json_to_dataframe(data)

    # 写到当前工作目录，供 Gradio File 组件下载
    csv_path = "synthetic_dataset.csv"
    df.to_csv(csv_path, index=False)

    # 返回：JSON 视图 + CSV 路径
    return data, csv_path


# Blocks：多区块自定义布局（比 Interface 更灵活）
with gr.Blocks() as demo:

    # 标题（Gradio Markdown 字符串保持原样，避免改 UI 文案）
    gr.Markdown("# 🧠 Synthetic Dataset Generator Studio")

    # ---------- 文本合成区 ----------
    gr.Markdown("## 📊 Text Synthetic Dataset Generator")

    # 任务类型下拉：键必须与 prompts 字典一致
    task_dropdown = gr.Dropdown(
        ["Customer Support", "Product Reviews", "Meeting Summary", "QA Dataset"],
        label="Select Dataset Type"
    )

    # 模型展示名下拉：键来自 models.keys()
    model_dropdown = gr.Dropdown(
        list(models.keys()),
        label="Select Model"
    )

    # 样本数量滑块：1~5，默认 2
    num_samples = gr.Slider(1, 5, value=2, step=1, label="Number of Samples")

    # 输出：校验后的 JSON 列表 + 可下载 CSV
    output_json = gr.JSON(label="Validated JSON Output")
    csv_output = gr.File(label="Download CSV")

    generate_btn = gr.Button("Generate Dataset")

    # 点击 → run_generator；show_progress 显示进度条
    generate_btn.click(
        run_generator,
        inputs=[task_dropdown, model_dropdown, num_samples],
        outputs=[output_json, csv_output],
        show_progress=True
    )

    # ---------- 分词器检查区 ----------
    gr.Markdown("## 🔍 Tokenizer Inspection")

    token_input = gr.Textbox(label="Enter Text")
    token_output = gr.JSON(label="Tokenizer Output")

    inspect_btn = gr.Button("Inspect Tokens")

    # 复用上面的 model_dropdown，对同一模型做 tokenize
    inspect_btn.click(
        inspect_tokenizer,
        inputs=[model_dropdown, token_input],
        outputs=token_output,
        show_progress=True
    )

    # ---------- 图像合成区 ----------
    gr.Markdown("## 🖼 Synthetic Image Dataset Generator")

    image_prompt = gr.Textbox(label="Image Prompt")
    num_images = gr.Slider(1, 3, value=1, step=1, label="Number of Images")
    image_gallery = gr.Gallery(label="Generated Images")

    image_btn = gr.Button("Generate Images")

    image_btn.click(
        generate_synthetic_image_dataset,
        inputs=[image_prompt, num_images],
        outputs=image_gallery,
        show_progress=True
    )

# 启动 Gradio 应用（Colab 会给出可访问链接）
demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5d818ebea3cc01fd74.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
